In [3]:
import os
import random
import json

import numpy as np
import pandas as pd

import scanpy as sc
import anndata as ad

import mygene
import pickle


import openai

In [4]:
from collections import Counter

In [5]:
data_dir = "/scratch/ahakobyan/single_cellm_data/"
project_dir = '/users/anna.hakobyan/projects/single-cellm/'

In [6]:
from functions import *

### Loading the data

In [7]:
data_dir + "archs4_geo"

'/scratch/ahakobyan/single_cellm_data/archs4_geo'

In [8]:
os.path.isfile(data_dir + "archs4_geo/annotations.json")

True

In [9]:
annotation_file = data_dir + "archs4_geo/annotations.json"
 
with open(annotation_file) as json_file:
    annots = json.load(json_file)

In [10]:
processed_annotation_file = data_dir + "archs4_geo/processed_annotations.json"
 
with open(processed_annotation_file) as json_file:
    processed_annots = json.load(json_file)

In [11]:
sample_types = [elem["sample_type"] for key, elem in annots.items()]

sample_type_counts = Counter(sample_types)

# sample_type_counts ===== 
# Counter({'cell_line': 234119,
#          'tissue': 173790,
#          'primary_cells': 148331,
#          'in_vitro_differentiated_cells': 55550,
#          'stem_cells': 34860,
#          'induced_pluripotent_stem_cells': 7038})

# treatment = [elem["treatment"] for key, elem in annots.items()]

sample_groups = {}
for type in sample_type_counts:
    sample_groups[type] = []

for sampleid, elem in annots.items():
    sample_groups[elem["sample_type"]].append(sampleid)

In [12]:
random.seed(50)

# selected_sample_ids = []
# selected_sample_ids.extend(sample_groups["induced_pluripotent_stem_cells"])
# selected_sample_ids.extend(random.sample(sample_groups["cell_line"], 28600 ) )
# selected_sample_ids.extend(random.sample(sample_groups["tissue"], 28600 ) )
# selected_sample_ids.extend(random.sample(sample_groups["primary_cells"], 28600 ) )
# selected_sample_ids.extend(random.sample(sample_groups["in_vitro_differentiated_cells"], 28600 ) )
# selected_sample_ids.extend(random.sample(sample_groups["stem_cells"], 28600 ) )

# with open(data_dir + "selected_sample_ids.txt", 'w') as f:
#     for sid in selected_sample_ids:
#         f.write(sid + "\n")

with open(data_dir + "selected_sample_ids.txt", "r") as f:
    selected_sample_ids = [ elem.strip() for elem in f.readlines()]

In [13]:
gene_ranks = sc.read_h5ad(data_dir + "archs4_geo/gene_ranks/gene_ranks_csr.h5ad")

In [14]:
common_samples = gene_ranks.obs.index.intersection(selected_sample_ids)

In [15]:
with open(project_dir + "resources/archs4/smp_exp_keywords.pickle", 'rb') as picklefile:
    embed_keywords = pickle.load(picklefile)

### Setting up the system prompts

In [77]:
system_prompt = f"""You are a research AI assistant trained in molecular biology. You are given a description of a biological sample, such as the state/phenotype of a single cell, in the form of a free text sample annotation (provided through the GEO database), ontology terms associated with the annotation, top expressed genes and a description in the format of keywords and a keyword-score that describe the sample.

Your task is to design a conversation between a biomedical researcher and an AI system, conversing about a biological sample. The researcher initially knows very little about the provided biological sample and asks questions to learn as much about it as possible. The AI system has access to the transcriptomic state of the sample (in this scenario, you are provided with a textual interpretation of this state as indicated above) and responds to the questions as well as possible to satisfy the researcher’s curiosity. Both, questions and answers must be fully in accordance with the provided information about the sample. If the researcher asks a question that cannot be answered with the provided information, the AI system should indicate so. 

The researcher has an interest in molecular biology, immunology, cancer biology, etc.  Their questions should be diverse and in the natural tone of a scientist curious about the nature/biology of the sample  The answers should be in a tone that the research AI assistant has access to sample information, but the researcher does not. The research AI assistant's task is to help the researcher learn as much relevant information about the sample as possible. Do not use words or phrases that would be unnatural for a researcher to ask, for example, do not include sample names. Use simple and conversational language.

Identify the relevant questions that a researcher could ask based on the provided information.

Include diverse questions that inquire about sample description, the transcriptome, the tissue, top expressed genes, pathway activities, and similarities to other samples or tissues. Identify and ask other relevant questions that a researcher could ask about an RNA-seq sample. 

Only include questions that have definite answers.
(1) one can find the answer to the question in the input provided by the user and can answer confidently;
(2) one can confidently determine from the input if a statement is not true.
Do not ask any questions that cannot be answered confidently.


Do:
- ask interesting, diverse, and complex questions
- ask about the sample from the perspective of the researcher who does not know about the sample information
- provide relevant answers
- provide detailed answers to complex questions. Include reasoning steps to make the content more convincing and well-organized
- provide extensive sample descriptions
- keep a natural and professional tone
- include multiple paragraphs if necessary
- refer to the data as "this sample" or "this cell type"

Do not:
- do not express emotions (appreciation, gratitude, ...)
- do not include technical terms (e.g. ontology terms, similarity scores, etc.)
- do not include sample names
- do not in the question do not add the information type that contains the answer
- do not ask questions that cannot be answered given the provided sample information
- do not use category names
- do not use the cell line name
- do not use sample name

"""

### Setting up the client

In [78]:
# Configure the OpenAI library with the API secret and the custom base URL
openai.api_key = 'OdNFpEua0T5TNdd0'
openai.api_base = 'https://engage-projectors-du-seen.trycloudflare.com/v1'

mixtral_client = openai.OpenAI(api_key = openai.api_key, base_url = openai.api_base)

In [79]:
few_shot_samples = ["SRX185900", "SRX186162", "SRX188848"]

In [80]:
common_samples.intersection(embed_keywords).difference(few_shot_samples)

Index(['SRX186160', 'SRX186171', 'SRX188847', 'SRX195294', 'SRX195464',
       'SRX198056', 'SRX212298', 'SRX212595', 'SRX215397', 'SRX216192',
       'SRX218319'],
      dtype='object', name='experiment')

In [ ]:
5

In [81]:
%%time

for sample_id in common_samples.intersection(embed_keywords).difference(few_shot_samples):
    print(sample_id)
    user_input = get_user_input(sample_id, processed_annots, gene_ranks, embed_keywords)

    messages = [{"role": "system", "content": system_prompt}]
    
    ### Providing few-shots
    for sample in few_shot_samples[0:1]:

        with open(data_dir + "prompt_data/few_shot_examples/" + sample + "prompt.txt") as f:
            sample_context = f.read()

        with open(data_dir + "prompt_data/few_shot_examples/" + sample + "out.txt") as f:
            sample_response = f.read()

        messages.append ({"role": "user", "content": sample_context})
        messages.append ({"role": "assistant", "content": sample_response})

    messages.append({"role": "user", "content": user_input})
        

    response_1 = mixtral_client.chat.completions.create(
        model="mixtral",
        messages=messages,
        max_tokens=1000
    )
    with open(data_dir + "prompt_data/" + sample_id + "_prompt_mixtral_FS.txt", 'w') as f:
        f.write(user_input)
    
    with open(data_dir + "prompt_data/" + sample_id + "_out_mixtral_FS.txt", 'w') as f:
        f.write(response_1.choices[0].message.content)

SRX186160
SRX186171
SRX188847
SRX195294
SRX195464
SRX198056
SRX212298
SRX212595
SRX215397
SRX216192
SRX218319
CPU times: user 291 ms, sys: 32.8 ms, total: 324 ms
Wall time: 5min 25s
